<img src="https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aureliensalas/python_finance/blob/main/ml_crypto/assignment/banquier_algorithmique.ipynb)

# Le banquier algorithmique

**Travail en autonomie** · bloc 4 · à faire après la séance 8

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.

Ce notebook n'est pas noté. Il sert à vérifier, seul, que vous savez faire ce que la séance 8 a montré — et il se termine sur une question à laquelle aucun modèle ne répond.

## La commande

Vous êtes analyste crédit dans une banque. La direction vous confie les dossiers de **1 000 prêts passés** : pour chacun, ce qu'on savait au moment de l'accorder, et ce qui s'est passé ensuite — **remboursé**, ou **défaut**.

Elle veut un outil qui, pour un nouveau dossier, donne **une probabilité de défaut**, et une **règle** : à partir de quelle probabilité refuse-t-on ?

Et le comité d'éthique de la banque a glissé une note dans le dossier :

> *« A-t-on le droit — et a-t-on intérêt — à utiliser l'âge du client ? »*

**Prenez position maintenant, avant de calculer.** Notez vos deux réponses quelque part :

1. Parmi ces informations, **quelles sont les deux** qui prédisent le mieux un défaut ? La durée du prêt, son montant, l'absence de compte courant, un compte à découvert, une épargne faible, des incidents de paiement passés, un emploi récent.
2. Un banquier devrait-il pouvoir tenir compte de **l'âge** du client : oui, non, ça dépend ?

## Comment on va procéder

Comme d'habitude : une fonction dessine la fiche, vous calculez ce qui va dedans, **sous les noms de variables demandés**, et `afficher_scoring()` la remplit au fur et à mesure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

pd.set_option("display.max_rows", 12)
BASE_ML = "https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/ml_crypto/data/"


def verifier(nom, condition, indice=""):
    if condition:
        print("OK       -", nom)
    else:
        print("A REVOIR -", nom, ":", indice)

La cellule suivante contient la fonction qui dessine la fiche. **Vous n'avez pas à la lire** : exécutez-la, puis oubliez-la.

In [ ]:
ENCRE, GRIS, BLEU, ROUGE, VERT, ORANGE, FOND = "#1F2933", "#9AA5B1", "#1F4E79", "#B4341F", "#2E7D5B", "#B26B00", "#F5F7F9"
HALO = dict(facecolor="white", alpha=0.92, edgecolor="none", pad=2.2)
_vide = lambda v: v is None or (isinstance(v, float) and np.isnan(v))

NOMS = {"duree_mois": "durée du prêt (par mois)", "montant": "montant (par millier de DM)",
        "sans_compte": "pas de compte courant", "compte_negatif": "compte à découvert",
        "epargne_faible": "épargne faible", "incidents_passes": "incidents passés",
        "emploi_recent": "emploi récent ou sans emploi", "age": "âge (par année)"}


def _attente(a, titre):
    a.set_xticks([]); a.set_yticks([])
    for s in ("top", "right", "bottom", "left"): a.spines[s].set_visible(False)
    a.add_patch(plt.Rectangle((0, 0), 1, 1, transform=a.transAxes, facecolor=FOND,
                              edgecolor="#DCE2E8", lw=1, zorder=0))
    a.set_title(titre, loc="left", fontsize=10.5, color=GRIS, fontweight="bold")
    a.text(0.5, 0.5, "à compléter", transform=a.transAxes, fontsize=12, color=GRIS,
           ha="center", va="center", style="italic")


def _habiller(a, titre):
    a.set_title(titre, loc="left", fontsize=10.5, color=ENCRE, fontweight="bold")
    for s in ("top", "right"): a.spines[s].set_visible(False)
    a.tick_params(labelsize=7.5, colors=GRIS)
    a.title.set_position((-0.0, 1.0))


def _tuile(fig, case, titre, val, fmt, sous, coul=ENCRE):
    t = fig.add_subplot(case); t.axis("off")
    t.add_patch(plt.Rectangle((0, 0), 1, 1, transform=t.transAxes, color=FOND, zorder=0))
    t.add_patch(plt.Rectangle((0, 0.92), 1, 0.08, transform=t.transAxes,
                              color=GRIS if _vide(val) else BLEU, zorder=1))
    t.text(0.5, 0.72, titre, transform=t.transAxes, fontsize=6.8, color=GRIS, ha="center", fontweight="bold")
    if _vide(val):
        t.text(0.5, 0.36, "à compléter", transform=t.transAxes, fontsize=9.5, color=GRIS, ha="center", style="italic")
    else:
        t.text(0.5, 0.38, fmt(val), transform=t.transAxes, fontsize=14, color=coul, ha="center", fontweight="bold")
        t.text(0.5, 0.10, sous, transform=t.transAxes, fontsize=6.8, color=GRIS, ha="center")


def _pct(v):
    return ("%.0f %%" % (v * 100 if v <= 1 else v))


def dessiner_scoring(taux_defaut=None, defaut_sans_compte=None, defaut_incidents=None,
                     proba=None, seuil=None, matrice=None,
                     justesse=None, paresseuse=None, rappel=None, precision=None,
                     seuils=None, factures=None, meilleur_seuil=None, coefs=None,
                     refus_jeunes=None, refus_autres=None, refus_jeunes_age=None, refus_autres_age=None):

    fig = plt.figure(figsize=(11.5, 16.5), facecolor="white")
    gs = GridSpec(7, 6, figure=fig, height_ratios=[0.48, 0.72, 1.55, 1.55, 0.30, 1.35, 0.35],
                  hspace=0.62, wspace=0.55, left=0.06, right=0.965, top=0.975, bottom=0.03)

    # ---------------------------------------------------------------- bandeau
    a = fig.add_subplot(gs[0, :]); a.axis("off")
    a.add_patch(plt.Rectangle((0, 0), 1, 1, transform=a.transAxes, color=BLEU, zorder=0))
    a.text(0.022, 0.60, "Scoring crédit", transform=a.transAxes, fontsize=22, color="white", fontweight="bold", va="center")
    a.text(0.022, 0.18, "1 000 dossiers · une banque d'Allemagne du Sud · 1973-1975", transform=a.transAxes,
           fontsize=8.5, color="#C3D0DC", va="center")
    a.text(0.978, 0.50, "LE BANQUIER ALGORITHMIQUE", transform=a.transAxes, fontsize=11, color="white",
           ha="right", va="center", fontweight="bold", alpha=0.9)

    # ---------------------------------------------------------------- tuiles
    c = [gs[1, k] for k in range(6)]
    _tuile(fig, c[0], "TAUX DE DÉFAUT", taux_defaut, lambda v: "%.0f %%" % v, "dans les 1 000 dossiers")
    _tuile(fig, c[1], "JUSTESSE À 0,5", justesse,
           lambda v: _pct(v), "" if _vide(paresseuse) else "tout accepter : " + _pct(paresseuse))
    _tuile(fig, c[2], "RAPPEL À 0,5", rappel, _pct, "des mauvais payeurs refusés", ORANGE if not _vide(rappel) and rappel < 0.6 else ENCRE)
    _tuile(fig, c[3], "PRÉCISION À 0,5", precision, _pct, "des refus justifiés")
    _tuile(fig, c[4], "SEUIL RETENU", meilleur_seuil, lambda v: ("%.2f" % v).replace(".", ","), "la facture la plus basse", VERT)
    fmin = None if factures is None else min(factures)
    _tuile(fig, c[5], "FACTURE", fmin, lambda v: "%.0f" % v,
           "" if factures is None else "contre %.0f au seuil 0,5" % factures[0], VERT)

    # ---------------------------------------------------------------- probabilités
    a = fig.add_subplot(gs[2, :3])
    if proba is None:
        _attente(a, "La probabilité de défaut, dossier par dossier")
    else:
        a.hist(proba, bins=20, range=(0, 1), color=BLEU, alpha=0.8, edgecolor="white")
        if not _vide(seuil):
            a.axvline(seuil, color=ROUGE, lw=1.8, ls="--")
            a.text(seuil, a.get_ylim()[1] * 0.95, "  seuil %s → refus" % ("%.2f" % seuil).replace(".", ","),
                   color=ROUGE, fontsize=8.5, fontweight="bold", va="top", bbox=HALO)
        a.set_xlabel("probabilité de défaut", fontsize=8)
        _habiller(a, "La probabilité de défaut, dossier par dossier")

    # ---------------------------------------------------------------- matrice
    a = fig.add_subplot(gs[2, 3:])
    if matrice is None:
        _attente(a, "La matrice de confusion")
    else:
        m = np.asarray(matrice)
        a.imshow([[1, 0], [0, 1]], cmap=plt.matplotlib.colors.ListedColormap(["#F6E3DF", "#E1EFE8"]), aspect="auto")
        noms = [["accordé, remboursé\n(vrai négatif)", "refusé, aurait remboursé\n(faux positif)"],
                ["accordé, défaut\n(faux négatif)", "refusé, aurait fait défaut\n(vrai positif)"]]
        for i in range(2):
            for j in range(2):
                a.text(j, i - 0.08, str(m[i, j]), ha="center", va="center", fontsize=20, fontweight="bold",
                       color=VERT if i == j else ROUGE)
                a.text(j, i + 0.26, noms[i][j], ha="center", va="center", fontsize=7.5, color=ENCRE)
        a.set_xticks([0, 1]); a.set_xticklabels(["accordé", "refusé"], fontsize=8.5)
        a.set_yticks([0, 1]); a.set_yticklabels(["a remboursé", "a fait défaut"], fontsize=8.5)
        a.tick_params(length=0)
        for s in a.spines.values(): s.set_visible(False)
        a.set_title("La matrice de confusion" + ("" if _vide(seuil) else " au seuil %s" % ("%.2f" % seuil).replace(".", ",")),
                    loc="left", fontsize=10.5, color=ENCRE, fontweight="bold")

    # ---------------------------------------------------------------- facture
    a = fig.add_subplot(gs[3, :3])
    if seuils is None or factures is None:
        _attente(a, "La facture selon le seuil : 5 × faux négatifs + faux positifs")
    else:
        a.plot(seuils, factures, color=BLEU, lw=1.8, marker="o")
        if not _vide(meilleur_seuil):
            k = list(seuils).index(meilleur_seuil)
            a.scatter([meilleur_seuil], [factures[k]], s=110, color=VERT, zorder=5)
            a.annotate("  le creux", (meilleur_seuil, factures[k]), color=VERT, fontsize=9, fontweight="bold", va="top")
        a.axvline(1 / 6, color=GRIS, lw=1, ls=":")
        a.text(1 / 6, a.get_ylim()[1], " 1/6", color=GRIS, fontsize=8, va="top")
        a.set_xlabel("seuil de refus", fontsize=8)
        _habiller(a, "La facture selon le seuil : 5 × faux négatifs + faux positifs")

    # ---------------------------------------------------------------- coefficients
    a = fig.add_subplot(gs[3, 4:])
    if coefs is None:
        _attente(a, "La grille : ce qui pèse dans la décision")
    else:
        s = pd.Series(coefs).sort_values()
        a.barh([NOMS.get(n, n) for n in s.index], s.values,
               color=[ROUGE if v > 0 else VERT for v in s.values], height=0.6)
        a.axvline(0, color=ENCRE, lw=0.8)
        a.tick_params(axis="y", labelsize=8)
        a.set_xlabel("← baisse le risque      augmente le risque →", fontsize=8)
        _habiller(a, "La grille : ce qui pèse dans la décision")

    # ---------------------------------------------------------------- comité d'éthique
    a = fig.add_subplot(gs[4, :]); a.axis("off")
    a.plot([0, 1], [0.35, 0.35], transform=a.transAxes, color="#DCE2E8", lw=1.2, zorder=0)
    a.text(0.0, 0.35, "  LE COMITÉ D'ÉTHIQUE  ", transform=a.transAxes, fontsize=11, color=BLEU,
           va="center", fontweight="bold", bbox=dict(facecolor="white", edgecolor="none", pad=1))
    a.text(1.0, 0.35, "  a-t-on le droit, et a-t-on intérêt, à utiliser l'âge ?", transform=a.transAxes,
           fontsize=8.5, color=GRIS, ha="right", va="center", style="italic",
           bbox=dict(facecolor="white", edgecolor="none", pad=1))

    a = fig.add_subplot(gs[5, 1:3])
    if defaut_sans_compte is None or defaut_incidents is None:
        _attente(a, "Qui fait défaut ? (sur les 1 000 dossiers)")
    else:
        noms = ["avec compte courant", "sans compte courant", "sans incident passé", "avec incidents passés"]
        vals = [defaut_sans_compte[0], defaut_sans_compte[1], defaut_incidents[0], defaut_incidents[1]]
        vals = [v * 100 if v <= 1 else v for v in vals]
        a.barh(noms[::-1], vals[::-1], color=[ROUGE, GRIS, ROUGE, GRIS], height=0.6)
        for i, v in enumerate(vals[::-1]):
            a.text(v, i, "  %.0f %%" % v, va="center", fontsize=8.5)
        a.set_xlim(0, 100); a.tick_params(axis="y", labelsize=8)
        a.set_xlabel("% de défauts", fontsize=8)
        _habiller(a, "Qui fait défaut ? (sur les 1 000 dossiers)")

    a = fig.add_subplot(gs[5, 3:])
    if _vide(refus_jeunes) or _vide(refus_autres):
        _attente(a, "Part des dossiers refusés, au seuil retenu")
    else:
        x = np.arange(2)
        a.bar(x - 0.2, [refus_jeunes * 100, refus_autres * 100], width=0.38, color=BLEU, label="modèle sans l'âge")
        if not (_vide(refus_jeunes_age) or _vide(refus_autres_age)):
            a.bar(x + 0.2, [refus_jeunes_age * 100, refus_autres_age * 100], width=0.38, color=ORANGE, label="modèle avec l'âge")
        for xx, v in zip(list(x - 0.2) + list(x + 0.2),
                         [refus_jeunes, refus_autres, refus_jeunes_age, refus_autres_age]):
            if not _vide(v):
                a.text(xx, v * 100 + 1.5, "%.0f %%" % (v * 100), ha="center", fontsize=8.5)
        a.set_xticks(x); a.set_xticklabels(["moins de 25 ans", "25 ans et plus"], fontsize=8.5)
        a.set_ylim(0, 105); a.legend(loc="upper right", frameon=False, fontsize=8)
        _habiller(a, "Part des dossiers refusés, au seuil retenu")

    a = fig.add_subplot(gs[6, :]); a.axis("off")
    a.text(0.5, 0.5, "Ce que la fiche ne dit pas : pourquoi. Une régression mesure « ça va ensemble », jamais « ça cause ».",
           transform=a.transAxes, ha="center", va="center", fontsize=9, color=GRIS, style="italic")


def afficher_scoring():
    """Va chercher vos variables par leur nom, et dessine la fiche avec tout ce qui existe déjà."""
    g = globals()
    lire = lambda nom: g.get(nom)
    retenu = "meilleur_seuil" in g and "matrice_retenue" in g
    dessiner_scoring(
        taux_defaut=lire("taux_defaut"), defaut_sans_compte=lire("defaut_sans_compte"),
        defaut_incidents=lire("defaut_incidents"), proba=lire("proba"),
        seuil=lire("meilleur_seuil") if retenu else (0.5 if "matrice" in g else None),
        matrice=lire("matrice_retenue") if retenu else lire("matrice"),
        justesse=lire("justesse"), paresseuse=lire("paresseuse"), rappel=lire("rappel"), precision=lire("precision"),
        seuils=lire("seuils") if "factures" in g else None, factures=lire("factures"),
        meilleur_seuil=lire("meilleur_seuil"), coefs=lire("coefs"),
        refus_jeunes=lire("refus_jeunes"), refus_autres=lire("refus_autres"),
        refus_jeunes_age=lire("refus_jeunes_age"), refus_autres_age=lire("refus_autres_age"))
    plt.show()

## Les données

`credit.csv` : de **vrais crédits**, accordés par une banque d'Allemagne du Sud entre **1973 et 1975**. C'est la version **corrigée**, publiée en 2019, du « German Credit », le jeu de données le plus utilisé au monde pour enseigner le scoring. L'ancienne version contenait des **erreurs de codage restées vingt-cinq ans sans que personne ne les voie**. Retenez-le : ça servira à la fin.

| colonne | ce que c'est |
|---|---|
| `duree_mois` | la durée du prêt, en mois |
| `montant` | le montant, en milliers de deutschemarks |
| `sans_compte` | 1 si le client n'a pas de compte courant dans la banque |
| `compte_negatif` | 1 si son compte courant est à découvert |
| `epargne_faible` | 1 si son épargne est nulle, inconnue, ou sous 100 DM |
| `incidents_passes` | 1 s'il a eu des retards ou des incidents sur des crédits passés |
| `emploi_recent` | 1 s'il est sans emploi, ou dans son emploi depuis moins d'un an |
| `age` | son âge |
| `etranger` | 1 si c'est un travailleur étranger |
| `situation` | la case du formulaire de l'époque, sexe et situation familiale mélangés |
| `defaut` | **la cible** : 1 si le prêt a fait défaut |

> **Deux avertissements.** (1) **30 % de défauts, ce n'est pas la réalité** : la banque a exprès gardé beaucoup de mauvais dossiers dans l'échantillon, pour qu'il y en ait assez à étudier. (2) `age`, `etranger` et `situation` **n'entrent pas dans le modèle** au départ. On les garde pour le comité d'éthique.

In [ ]:
credit = pd.read_csv(BASE_ML + "credit.csv")
print(credit.shape)
credit.head()

In [ ]:
afficher_scoring()

La fiche est grise. En haut, six chiffres. Au milieu, l'outil : les probabilités, la matrice, la facture, la grille. En bas, **le comité d'éthique**.

---

# Premier volet — l'outil

## Étape 1 — Regarder avant de modéliser

Avant tout modèle, on regarde les données. Toujours.

**Ce qu'on cherche.**

- `taux_defaut` : la part de défauts dans les 1 000 dossiers, **en %** ;
- `defaut_sans_compte` : le taux de défaut **selon** `sans_compte` — une Series à deux valeurs, pour 0 et pour 1 ;
- `defaut_incidents` : pareil selon `incidents_passes`.

**Où c'était.** La moyenne d'une colonne de 0 et de 1 est une part (bloc 3). `groupby(...)[...].mean()` (bloc 2).

In [ ]:
verifier("taux_defaut en %", abs(taux_defaut - 30) < 1e-9, "la moyenne de la colonne defaut, fois 100")
verifier("defaut_sans_compte", abs(defaut_sans_compte[1] - credit.query("sans_compte == 1")["defaut"].mean()) < 1e-9, "credit.groupby('sans_compte')['defaut'].mean()")
verifier("defaut_incidents", abs(defaut_incidents[1] - credit.query("incidents_passes == 1")["defaut"].mean()) < 1e-9, "credit.groupby('incidents_passes')['defaut'].mean()")
afficher_scoring()

**Lecture.** Sans compte courant : **49 %** de défauts, contre 23 %. Avec des incidents passés : **60 %**, contre 27 %. On voit déjà ce que le modèle va trouver.

Vos deux paris du début étaient-ils dans le haut ? Vous pouvez vérifier les autres variables de la même façon, dans une cellule à vous.

## Étape 2 — Le découpage, et cette fois on mélange

**Une question, avant de coder.** Pendant tout le bloc, on a interdit de mélanger. Pourquoi est-ce permis ici ?

<details><summary><b>La réponse</b> (réfléchissez d'abord)</summary>

Les cryptos étaient des **dates** : mélanger, c'était apprendre sur l'avenir pour prévoir le passé. Ici, ce sont des **clients**, tous de la même période. Le dossier n° 17 ne vient ni avant ni après le dossier n° 512. Mélanger garantit simplement que l'entraînement et le test se ressemblent.

**La règle n'a jamais été « ne jamais mélanger ». C'était « ne jamais laisser le modèle voir l'avenir ».**
</details>

**Ce qu'on cherche.**

- `variables` : la **liste** des sept noms de colonnes du modèle — tout sauf `age`, `etranger`, `situation` et `defaut` ;
- `train, test` : le découpage de **toute la table** `credit`, avec `test_size=0.25` et `random_state=0`, **sans** `shuffle=False` ;
- `X_train, y_train, X_test, y_test` : les variables et la cible, dans chacune des deux tables.

**Deux gestes nouveaux, minuscules.**

- Donnée **une seule table**, `train_test_split` en rend **deux** : `train, test = train_test_split(credit, ...)`. On garde ainsi l'âge dans `test`, pour le comité d'éthique.
- `random_state=0` : le tirage est au hasard, mais le même pour tout le monde. C'est ce qui permet aux `verifier` de connaître vos chiffres.

In [ ]:
_tr, _te = train_test_split(credit, test_size=0.25, random_state=0)
verifier("les sept variables", sorted(variables) == sorted(["duree_mois", "montant", "sans_compte", "compte_negatif", "epargne_faible", "incidents_passes", "emploi_recent"]), "ni age, ni etranger, ni situation, ni defaut")
verifier("750 et 250", len(train) == 750 and len(test) == 250, "test_size=0.25")
verifier("le même tirage que tout le monde", list(test.index) == list(_te.index), "random_state=0, et pas de shuffle=False")
verifier("X et y", list(X_train.columns) == variables and len(y_test) == 250, "train[variables], train['defaut'], puis pareil sur test")

## Étape 3 — Le modèle et ses probabilités

**Ce qu'on cherche.** `modele`, une `LogisticRegression` ajustée sur l'entraînement ; `proba`, la probabilité de **défaut** de chaque dossier du test.

**Où c'était.** Séance 8, sections 1 et 2.

**Les pièges.**

- `predict_proba` sort **deux colonnes** : la probabilité de 0 (remboursé), puis celle de 1 (défaut). C'est la seconde qu'on veut : `[:, 1]`.
- Si vous voyez un avertissement *ConvergenceWarning*, ce n'est pas une erreur : vous avez oublié `max_iter=1000`.

In [ ]:
_m = LogisticRegression(max_iter=1000).fit(X_train, y_train)
verifier("une probabilité par dossier", len(proba) == 250, "predict_proba sur X_test")
verifier("la probabilité de défaut", abs(proba - _m.predict_proba(X_test)[:, 1]).max() < 1e-6, "la colonne 1 : [:, 1] ; si presque tout est au-dessus de 0,5, vous avez pris la colonne 0")
afficher_scoring()

**Lecture.** Les probabilités vont de **5 %** pour le dossier le plus sûr à **81 %** pour le plus risqué. Le modèle ne décide rien : **il classe**. La décision, c'est vous.

## Étape 4 — Décider à 50 %

On commence par le seuil de la séance 8, section 3 : on refuse si la probabilité de défaut dépasse 0,5.

**Les quatre cases, dans les mots du banquier.** « Positif », ici, c'est « défaut ».

| | le client a remboursé | le client a fait défaut |
|---|---|---|
| **prêt accordé** | vrai négatif | **faux négatif** : on a prêté à un mauvais payeur |
| **prêt refusé** | **faux positif** : un bon client, parti chez le concurrent | vrai positif |

| mesure | la question du banquier |
|---|---|
| **rappel** | De tous les clients qui ont fait défaut, **combien ai-je refusés** ? |
| **précision** | De tous les clients que j'ai refusés, **combien auraient vraiment fait défaut** ? |

**Ce qu'on cherche.** `decision`, `matrice`, les quatre cases `vn, fp, fn, vp`, puis `justesse`, `rappel`, `precision`, et `paresseuse` : la justesse de la banque qui accepte tout le monde.

**Où c'était.** Séance 8, sections 3 à 6.

In [ ]:
_d = (proba >= 0.5).astype(int)
verifier("la matrice", (matrice == confusion_matrix(y_test, _d)).all(), "confusion_matrix(y_test, decision), dans cet ordre")
verifier("justesse", abs(justesse - accuracy_score(y_test, _d)) < 1e-9, "(vn + vp) / len(y_test)")
verifier("rappel", abs(rappel - recall_score(y_test, _d)) < 1e-9, "le rappel se lit sur la ligne des vrais défauts : vp / (vp + fn)")
verifier("precision", abs(precision - precision_score(y_test, _d)) < 1e-9, "la précision se lit sur la colonne des refus : vp / (vp + fp)")
verifier("paresseuse", abs(paresseuse - (1 - y_test.mean())) < 1e-9, "la part de clients qui ont remboursé : 1 - y_test.mean()")
afficher_scoring()

**Lecture.** Justesse **76 %**, contre **67 %** pour la banque qui accepte tout le monde. Pas mal.

Mais regardez le **rappel : 39 %**. Sur les 83 mauvais payeurs du test, **51 ont reçu leur prêt**. La précision, elle, est haute : quand le modèle refuse, il a presque toujours raison. C'est l'alarme timide de la séance 8 — et pour une banque, c'est cher.

## Étape 5 — Ce que coûte chaque erreur

**Le prix, et il n'est pas inventé.** La documentation de ces données, rédigée à l'époque, fixe elle-même les coûts : **accorder un prêt à un mauvais payeur coûte 5 fois plus cher que refuser un bon client.** D'un côté, le capital perdu ; de l'autre, une marge d'intérêts manquée.

La **facture** d'un seuil, c'est donc : **5 × faux négatifs + 1 × faux positifs**.

**D'abord, un calcul de tête.** Au seuil théorique, le coût attendu d'un refus égale celui d'une acceptation : p × 5 = (1 − p) × 1. Calculez `seuil_theorique`, la probabilité p qui l'équilibre. *(Indice : c'est 1 divisé par 1 + 5.)*

**Ensuite, la boucle.** Pour chaque seuil de la liste, elle refait la décision et la matrice. **Complétez la ligne qui ajoute la facture** à la liste `factures`. Puis `meilleur_seuil` : le seuil dont la facture est la plus basse. Mettez les factures dans une Series avec les seuils en index, et demandez-lui `idxmin()` — le geste du bloc 3.

Enfin, `matrice_retenue` : la matrice de confusion au meilleur seuil.

**Où c'était.** Séance 8, section 7 : la même boucle, avec la facture de la plateforme.

In [ ]:
seuil_theorique = ...
seuils = [0.5, 0.4, 0.3, 0.25, 0.2, 0.17, 0.15, 0.1, 0.05]
factures = []
for s in seuils:
    vn_s, fp_s, fn_s, vp_s = confusion_matrix(y_test, (proba >= s).astype(int)).ravel()
    # à vous : une ligne, qui ajoute la facture à la liste

meilleur_seuil = ...
matrice_retenue = ...
print(seuil_theorique, meilleur_seuil)
print(pd.Series(factures, index=seuils))

In [ ]:
_f = [5 * confusion_matrix(y_test, (proba >= s).astype(int)).ravel()[2] + confusion_matrix(y_test, (proba >= s).astype(int)).ravel()[1] for s in seuils]
verifier("seuil_theorique", abs(seuil_theorique - 1 / 6) < 1e-9, "p × 5 = (1 − p) × 1, donc p = 1 / 6")
verifier("les factures", len(factures) == 9 and [int(x) for x in factures] == [int(x) for x in _f], "5 * fn_s + fp_s : les faux négatifs coûtent 5, les faux positifs 1")
verifier("meilleur_seuil", meilleur_seuil == seuils[_f.index(min(_f))], "pd.Series(factures, index=seuils).idxmin()")
verifier("matrice_retenue", (matrice_retenue == confusion_matrix(y_test, (proba >= meilleur_seuil).astype(int))).all(), "confusion_matrix au meilleur seuil")
afficher_scoring()

**Lecture.** Facture **265** au seuil de 0,5 ; **135** au seuil de **0,17**. On **divise la facture par deux sans toucher au modèle**. Et le creux tombe exactement là où le calcul de tête l'annonçait : 1/6 ≈ 0,17, le trait pointillé sur la courbe.

Regardez la nouvelle matrice : 73 des 83 mauvais payeurs sont refusés. Le prix : 85 bons clients refusés aussi.

**Au meilleur seuil, la banque refuse près de deux dossiers sur trois. Est-ce réaliste ?** Relisez le premier avertissement de la section « Les données ».

*[votre réponse]*

## Étape 6 — Lire la grille

**Ce qu'on cherche.** `coefs` : une Series qui contient **un coefficient par variable**, avec le nom des variables en index. Les coefficients sont dans `modele.coef_[0]`.

**Où c'était.** `coef_` (séance 7, section 2) ; créer une Series avec un index (bloc 1 et bloc 2).

In [ ]:
verifier("coefs", list(coefs.index) == variables and abs(coefs["sans_compte"] - modele.coef_[0][variables.index("sans_compte")]) < 1e-9, "pd.Series(modele.coef_[0], index=variables)")
afficher_scoring()

**Lecture.** En rouge, ce qui **augmente** le risque : pas de compte courant, un découvert, des incidents passés, une épargne faible, un emploi récent, une durée longue. Le montant ne compte presque plus une fois la durée connue.

**Un client refusé demande pourquoi. Que lui répondez-vous ?** Et pourquoi une banque tient-elle à pouvoir répondre ?

*[votre réponse]*

---

# Second volet — le comité d'éthique

> Le modèle marche. Reste la question du comité. **Elle n'a pas de `verifier`** pour la réponse : il n'y a pas de bonne case à cocher, seulement des arguments. Les `verifier` ne contrôlent que les chiffres, pour que le débat parte de chiffres justes.

## Étape 7 — L'âge

**Ce qu'on cherche, en trois temps.**

**1.** `defaut_jeunes` et `defaut_autres` : le taux de défaut, en %, des moins de 25 ans et des autres, sur toute la table `credit`.

**2.** `modele_age` : le même modèle, **avec** `age` en plus des sept variables, ajusté sur l'entraînement ; `proba_age`, ses probabilités sur le test ; et `facture_age`, sa facture au seuil retenu, `meilleur_seuil`.

**3.** Qui est refusé ? Au seuil retenu, la part de dossiers refusés chez les jeunes et chez les autres, **pour les deux modèles** : `refus_jeunes`, `refus_autres` (sans l'âge), `refus_jeunes_age`, `refus_autres_age` (avec). La cellule d'après prépare le terrain : elle range les deux probabilités dans la table `test`.

**Où c'était.** `query` (bloc 2), le modèle et la facture (étapes 3 et 5). Une comparaison sur une colonne, puis `mean()`, donne une part (bloc 3).

In [ ]:
_ma = LogisticRegression(max_iter=1000).fit(train[variables + ["age"]], y_train)
_pa = _ma.predict_proba(test[variables + ["age"]])[:, 1]
_ra = confusion_matrix(y_test, (_pa >= meilleur_seuil).astype(int)).ravel()
verifier("defaut_jeunes et defaut_autres", abs(defaut_jeunes - credit.query("age < 25")["defaut"].mean() * 100) < 1e-9 and abs(defaut_autres - credit.query("age >= 25")["defaut"].mean() * 100) < 1e-9, "query('age < 25'), puis la moyenne de defaut, fois 100")
verifier("proba_age", abs(proba_age - _pa).max() < 1e-6, "variables + ['age'] : on ajoute une colonne à la liste")
verifier("facture_age", int(facture_age) == int(5 * _ra[2] + _ra[1]), "5 × faux négatifs + faux positifs, au seuil meilleur_seuil")

In [ ]:
test = test.copy()
test["proba"] = proba
test["proba_age"] = proba_age
test[["age", "proba", "proba_age"]].head()

In [ ]:
_j, _o = test.query("age < 25"), test.query("age >= 25")
verifier("sans l'âge", abs(refus_jeunes - (_j["proba"] >= meilleur_seuil).mean()) < 1e-9 and abs(refus_autres - (_o["proba"] >= meilleur_seuil).mean()) < 1e-9, "(test.query('age < 25')['proba'] >= meilleur_seuil).mean()")
verifier("avec l'âge", abs(refus_jeunes_age - (_j["proba_age"] >= meilleur_seuil).mean()) < 1e-9 and abs(refus_autres_age - (_o["proba_age"] >= meilleur_seuil).mean()) < 1e-9, "la même chose avec la colonne proba_age")
afficher_scoring()

**Les chiffres du débat.**

| | refus des moins de 25 ans | refus des 25 ans et plus | facture |
|---|---|---|---|
| modèle **sans** l'âge | **77 %** | 61 % | 135 |
| modèle **avec** l'âge | 82 % | 60 % | 141 |

Et le défaut réel : **41 %** chez les moins de 25 ans, 28 % chez les autres.

**Deux constats, qui font tout le débat.**

- **Ajouter l'âge ne rapporte rien** : la facture ne baisse pas. (Sur 200 découpages au hasard, l'écart moyen est nul.)
- **Retirer l'âge ne protège pas les jeunes** : sans lui, ils sont quand même refusés bien plus souvent. Comment le modèle fait-il, sans connaître leur âge ? La cellule suivante répond.

In [ ]:
credit.groupby(credit["age"] < 25)[["emploi_recent", "sans_compte", "epargne_faible"]].mean().round(2)

La ligne `True`, ce sont les moins de 25 ans. Ils ont plus souvent un emploi récent (36 % contre 21 %), plus souvent pas de compte courant (38 % contre 26 %). **Le modèle retrouve l'âge par d'autres chemins.** Ces variables, qui portent l'information d'une variable absente, s'appellent des **proxys**.

**Les jeunes font-ils défaut *parce qu'ils sont jeunes*, ou parce qu'ils ont un emploi récent et pas d'épargne ?** Les données permettent-elles de trancher ?

*[votre réponse]*

**Refuser plus souvent les jeunes parce qu'ils ont moins d'épargne, est-ce de la discrimination ?** Et si retirer la variable ne suffit pas, que faudrait-il faire ?

*[votre réponse]*

## Étape 8 — Le sexe et la nationalité

**Ce qu'on cherche.** `situations` : le nombre de dossiers dans chaque catégorie de la colonne `situation`. Puis `nb_etrangers` : le nombre de travailleurs étrangers, et `defauts_etrangers` : combien d'entre eux ont fait défaut.

**Où c'était.** `value_counts` (bloc 2), `query` et `sum` (bloc 2).

In [ ]:
verifier("situations", situations.sum() == 1000 and len(situations) == 4, "credit['situation'].value_counts()")
verifier("nb_etrangers", nb_etrangers == credit["etranger"].sum(), "len(credit.query('etranger == 1'))")
verifier("defauts_etrangers", defauts_etrangers == credit.query("etranger == 1")["defaut"].sum(), "la somme de la colonne defaut, chez les étrangers")

**Ce que vous trouvez.**

- **Le sexe n'existe pas comme variable.** Le formulaire de 1973 avait une case « femme non célibataire, **ou** homme célibataire ». On ne peut pas séparer les femmes des hommes. Et la case elle-même dit quelque chose de l'époque : pour une femme, le statut marital comptait assez pour être demandé — et pas séparément du sexe.
- **37 travailleurs étrangers sur 1 000**, dont 4 défauts.

**Que peut-on conclure sur les étrangers à partir de 37 dossiers ?** Et que vous dit le fait que, pendant vingt-cinq ans, des milliers de cours et d'articles ont utilisé ces données avec la variable « étranger » **inversée** ?

*[votre réponse]*

---

# Pour finir

## Le guichet

Votre outil, en vrai. Réglez un dossier avec les curseurs et les cases : le modèle donne la probabilité de défaut, et la règle — votre `meilleur_seuil` — donne le verdict. Essayez de trouver **le plus petit changement** qui fait basculer un refus en accord.

In [ ]:
def guichet(duree_mois=24, montant=3.0, sans_compte=False, compte_negatif=False,
            epargne_faible=True, incidents_passes=False, emploi_recent=False):
    """Un dossier, une probabilité, un verdict — avec votre modèle et votre seuil."""
    dossier = pd.DataFrame([[duree_mois, montant, int(sans_compte), int(compte_negatif),
                             int(epargne_faible), int(incidents_passes), int(emploi_recent)]], columns=variables)
    p = modele.predict_proba(dossier)[0, 1]
    verdict = "REFUSÉ" if p >= meilleur_seuil else "ACCORDÉ"
    print(f"Probabilité de défaut : {p:.0%}   (seuil : {meilleur_seuil:.0%})")
    print(f"Verdict : {verdict}")


guichet(duree_mois=12, montant=2.0)

In [ ]:
import ipywidgets as widgets

widgets.interact(guichet,
                 duree_mois=widgets.IntSlider(value=24, min=6, max=72, step=6, description="durée (mois)"),
                 montant=widgets.FloatSlider(value=3.0, min=0.5, max=15.0, step=0.5, description="montant (k DM)"),
                 sans_compte=widgets.Checkbox(value=False, description="pas de compte courant"),
                 compte_negatif=widgets.Checkbox(value=False, description="compte à découvert"),
                 epargne_faible=widgets.Checkbox(value=True, description="épargne faible"),
                 incidents_passes=widgets.Checkbox(value=False, description="incidents passés"),
                 emploi_recent=widgets.Checkbox(value=False, description="emploi récent"));

> Si les curseurs ne s'affichent pas, ce n'est pas grave : appelez `guichet(...)` directement, comme dans la cellule du dessus, en changeant les valeurs.

## La note au comité

Double-cliquez sur la cellule « votre réponse » et rédigez **cinq phrases au plus** : le seuil que vous recommandez et ce qu'il coûte ; les deux informations qui pèsent le plus ; et **votre position argumentée** sur l'utilisation de l'âge.

**Votre note au comité.**

*[votre réponse]*

## La morale, en trois lignes

1. **Le modèle classe ; la règle décide.** Le même modèle, au bon seuil, divise la facture par deux.
2. **Une régression dit « ça va ensemble », jamais « ça cause ».** Les jeunes font plus défaut ; pourquoi, les données ne le disent pas.
3. **Retirer une variable ne la fait pas disparaître.** Les autres la portent.

## Et après ?

Le cours se termine en classe, sur ce que vous venez de trouver : **corrélation n'est pas causalité**, les proxys, et ce que ça fait à l'intelligence artificielle quand elle décide pour des personnes.

## Avant de fermer

1. *Exécution → Redémarrer et tout exécuter*.
2. Toutes les cellules `verifier` affichent `OK`.
3. Les réponses en texte, et la note au comité, sont écrites.